In [1]:
!pip install pymanopt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.3 MB/s eta 0:00:00


In [2]:
%reset -f

In [3]:
import numpy as np
# import tensorflow as tf
import cupy as cp
# import jax.numpy as jnp
# import jax
# from jax import jit, lax
# from jax import device_put
# from jax.lib import xla_bridge

# import pymanopt
# from pymanopt.manifolds import Oblique
# from pymanopt.optimizers import TrustRegions
#from cupy.dtypes import float8_e4m3fn, float8_e5m2

import time
import gc

In [4]:
# import os

# # 2. Use CUB and cuTENSOR backends (Faster reductions/elementwise)
# os.environ["CUPY_ACCELERATORS"] = "cub,cutensor"

# # 3. Tune the Compiler
# # Tells the GPU compiler to optimize for the A100 architecture specifically
# os.environ["CUDA_CACHE_MAXSIZE"] = "2147483648" # 2GB Cache
# # os.environ["CUDA_CACHE_MAXSIZE"] = "4294967296" # 2GB Cache

# # # TF32 settings
# # # Allow CuPy/CUDA to use Tensor Cores for Float32
# # os.environ["NVIDIA_TF32_OVERRIDE"] = "1"

# # # Tell CuPy to allow TF32
# # cp.cuda.set_allocator(cp.cuda.MemoryPool().malloc)
# # # Note: CuPy enables TF32 by default on A100,
# # # but checking specific matrix math settings helps.

In [5]:
def reset_memory():
    # Delete all possible GPU arrays
    globals_ = list(globals().keys())
    for var in globals_:
        if var in ['cp', 'jnp', 'reset_memory']: continue
        if isinstance(globals()[var], (cp.ndarray, jnp.ndarray)):
            del globals()[var]

    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

In [6]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))
shift = 50/n

C = cp.random.randn(n,n)/n
C = C + C.T
cp.fill_diagonal(C, C.diagonal() + shift)

# generate_B.seed = 0
# B = generate_B.randn(k,n)
B = cp.random.randn(k,n)
B = B/cp.linalg.norm(B,axis = 0)

# cp_f8 = float8_e5m2
# cp_f8 = float8_e4m3fn

C_16 = cp.array(C,dtype = cp.float16)
B_16 = cp.array(B,dtype = cp.float16)

# C_16 = cp.array(C,dtype = cp.float32)
# B_16 = cp.array(B,dtype = cp.float32)

B_16.dot(C_16)
cp.cuda.Device().synchronize()
# 1000 is good enough do not exceed 1500
K16 = [100,100,100,100,100,100,100,100,100,100]
for i in range(len(K16)):
  cg0 = time.time()
  for t in range(K16[i]):
      B_16 = B_16.dot(C_16)
      B_16 = B_16/cp.linalg.norm(B_16,axis = 0)
  cp.cuda.Device().synchronize()
  cg1 = time.time()
  obj64 = cp.tensordot(B_16,B_16.dot(C))
  print("block time",cg1-cg0)
  print("obj 64",obj64)

block time 0.5405020713806152
obj 64 531.6973230093697
block time 0.1969280242919922
obj 64 532.241809611099
block time 0.19765257835388184
obj 64 532.3225296572825
block time 0.19910740852355957
obj 64 532.3427903573954
block time 0.1978626251220703
obj 64 532.3847027194122
block time 0.19890046119689941
obj 64 532.4179628539566
block time 0.19973111152648926
obj 64 532.4172037489952
block time 0.19869780540466309
obj 64 532.4158098804367
block time 0.19945430755615234
obj 64 532.4220838634528
block time 0.19828104972839355
obj 64 532.4466541140126


In [7]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))

repeat = 50

f_cg = np.zeros((repeat,10))
t_cg = np.zeros((repeat,10))

shift = 50/n

cp.random.seed(0)
for rep in range(repeat):
  print("----------------REP++",rep,"++---------------------------------------")
  # reset_memory()
  C = cp.random.randn(n,n)/n
  C = C + C.T
  cp.fill_diagonal(C, C.diagonal() + shift)

  # generate_B.seed = 0
  # B = generate_B.randn(k,n)
  B = cp.random.randn(k,n)
  B = B/cp.linalg.norm(B,axis = 0)

  C_16 = cp.array(C,dtype = cp.float16)
  B_16 = cp.array(B,dtype = cp.float16)
  dummy_res = B_16.dot(C_16)
  cp.cuda.Device().synchronize()

  # 1000 is good enough do not exceed 1500
  K16 = [100,100,100,100,100,100,100,100,100,100]
  for i in range(len(K16)):
    cg0 = time.time()
    for t in range(K16[i]):
        B_16 = B_16.dot(C_16)
        B_16 = B_16/cp.linalg.norm(B_16,axis = 0)
    cp.cuda.Device().synchronize()
    cg1 = time.time()

    # obj64 = cp.tensordot(B_16,B_16.dot(C))-shift *n
    obj64 = cp.tensordot(B_16,B_16.dot(C))
    print("block time",cg1-cg0)
    print("obj 64",obj64)
    f_cg[rep,i] = obj64
    t_cg[rep,i] = cg1-cg0

----------------REP++ 0 ++---------------------------------------
block time 0.19798064231872559
obj 64 531.7190841896908
block time 0.1993846893310547
obj 64 532.2654257038231
block time 0.19873332977294922
obj 64 532.3344742451318
block time 0.20006656646728516
obj 64 532.3448594568152
block time 0.19923686981201172
obj 64 532.3987452056917
block time 0.19799518585205078
obj 64 532.4273442027331
block time 0.19953203201293945
obj 64 532.4455113107463
block time 0.20015478134155273
obj 64 532.4497800026655
block time 0.19866275787353516
obj 64 532.426374958022
block time 0.19873976707458496
obj 64 532.3303992189484
----------------REP++ 1 ++---------------------------------------
block time 0.19763898849487305
obj 64 531.7626477379772
block time 0.2001640796661377
obj 64 532.3193421217854
block time 0.1987001895904541
obj 64 532.389553943762
block time 0.19954967498779297
obj 64 532.4102697529925
block time 0.19912409782409668
obj 64 532.4671114741876
block time 0.20010995864868164
ob

In [8]:
!mkdir -p /content/drive/MyDrive/result_folder
np.savez("/content/drive/MyDrive/result_folder/results_gpu_float16_final_A100.npz",f_cg = f_cg, t_cg = t_cg)

In [9]:
np.cumsum(t_cg.mean(axis = 0))

array([0.2012111 , 0.40354128, 0.6059616 , 0.81011816, 1.01296829,
       1.21564386, 1.41833176, 1.62129819, 1.82434099, 2.02722777])

In [10]:
f_cg.mean(axis = 0)

array([531.68868949, 532.23411249, 532.30792978, 532.32804459,
       532.33423616, 532.34876454, 532.34606844, 532.34711727,
       532.35493013, 532.36154753])

In [11]:
print(cp.__version__)

14.0.1


In [12]:
!nvidia-smi

Fri Mar  6 03:52:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   61C    P0             95W /  400W |   27918MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----